# Doublet vs triplet spatial modality — multimodal fusion benchmark

Take the **3 best multimodal models** from `final_sweep_benchmark.ipynb` (all `S3_mvtcae`, POE +
MVTCAE, `n_latent=20`, batch-mask `ft`) and re-benchmark each with the spatial modality swapped from
**doublet** colocalization (marker-pair join-count Z) to **triplet** colocalization (open-wedge Z,
`coloc_triplet_asinh5 = arcsinh(z/5)`; 171 curated triplets from `chunglu_triplets_analysis.ipynb`).

Doublet arm = the cached `final_sweep` runs (identical methodology). Triplet arm = depth-matched
runs (`sp-triplet nl20 L2/L3`) trained by the same driver (`sweep/train_final_sweep.py`, new
`--spatial triplet` option). Benchmarking = the ref-sweep pipeline: per-modality Moran's I + AUC vs
scVI specialist, joint cross-modality Moran's I, scib-metrics, PPC reconstruction.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json, subprocess
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc
import matplotlib.pyplot as plt, seaborn as sns

PIXELGEN_ROOT = '/home/projects/nyosef/zvise/PixelGen'
PIXELGEN_PKG  = PIXELGEN_ROOT + '/PixelGen'
for p in (PIXELGEN_ROOT, PIXELGEN_PKG):
    if p not in sys.path:
        sys.path.append(p)

from PixelGen.metrics import distr_autocorrelation_in_latent
from PixelGen.utils import build_triplet_asinh5_obsm, get_dense

NEW_DATA = Path(PIXELGEN_PKG) / 'New_Data'
CACHE    = NEW_DATA / 'cache'
FINAL    = CACHE / 'final_sweep'          # cached doublet-arm runs + scVI specialists
TRIP_OUT = CACHE / 'triplet_sweep'        # triplet-arm runs + triplet specialist
CHUNGLU  = NEW_DATA / 'results' / 'chunglu'
ADATA_PATH = CACHE / 'adata_cytovi_annotated_compat.h5ad'

ABUNDANCE_REP = 'arcsinh'                    # abundance features for autocorrelation (layer)
DOUBLET_REP   = 'spatial_asinh5_top500var'  # doublet spatial features (ref-sweep spatial rep)
TRIPLET_REP   = 'coloc_triplet_asinh5'      # triplet spatial features, arcsinh(z/5)
BIO_KEY, BATCH_KEY = 'cell_type_annot', 'cell_system'
PCA_KW = {'n_comps': 15}
TOL = 0.02

sns.set_style('whitegrid')
sc.set_figure_params(figsize=(5, 3), frameon=False)

adata = sc.read_h5ad(ADATA_PATH)
build_triplet_asinh5_obsm(adata, CHUNGLU)   # -> obsm['coloc_triplet'], obsm['coloc_triplet_asinh5']
print(adata.shape, '|', BIO_KEY, adata.obs[BIO_KEY].nunique(), '|', BATCH_KEY, adata.obs[BATCH_KEY].nunique())
print('spatial reps:', {k: adata.obsm[k].shape for k in [DOUBLET_REP, TRIPLET_REP]})

## Step 1 — train the triplet-arm runs (LSF GPU)

Depth-matched to the 3 best doublet runs: `S3_mvtcae · nl20 · ft` at `L2` and `L3`, spatial =
triplet. Same driver/pipeline as the ref sweep, written to a **separate** out-root
(`cache/triplet_sweep`) so `final_sweep_benchmark.ipynb` is untouched. Idempotent on `DONE`. Flip
`SUBMIT = True` to fire the jobs.

In [ ]:
SWEEP = NEW_DATA / 'sweep'
LOGS  = SWEEP / 'logs_triplet'; LOGS.mkdir(parents=True, exist_ok=True)
TRIP_OUT.mkdir(parents=True, exist_ok=True)

TRIPLET_CONFIGS = [(20, 2), (20, 3)]         # (n_latent, n_layers); arch=S3_mvtcae, spatial=triplet, bm=ft
SUBMIT = True                               # <-- flip to True to actually submit

for nl, L in TRIPLET_CONFIGS:
    rid = f'arch-S3_mvtcae__sp-triplet__nl{nl}__L{L}__bm-ft'
    if (TRIP_OUT / rid / 'DONE').exists():
        print('[skip DONE]', rid); continue
    cmd = ['bsub', '-J', f'trip-{rid}', '-q', 'long-gpu',
           '-gpu', 'num=1:j_exclusive=yes:gmem=16G',
           '-R', 'rusage[mem=64G]', '-R', 'affinity[thread*8]',
           '-o', str(LOGS / f'{rid}.out'), '-e', str(LOGS / f'{rid}.err'),
           '--', 'bash', str(SWEEP / 'run_final_sweep.lsf'),
           '--arch', 'S3_mvtcae', '--spatial', 'triplet', '--n-latent', str(nl),
           '--n-layers', str(L), '--batch-mask', 'ft', '--out-root', str(TRIP_OUT)]
    if SUBMIT:
        subprocess.run(cmd, check=True); print('[submitted]', rid)
    else:
        print(' '.join(cmd))
print('\n' + ('submitted' if SUBMIT else 'DRY-RUN — set SUBMIT=True to fire'))

## Step 2 — specialist anchors (per-modality ceilings)

Three vanilla-scVI specialists give the per-modality Moran's-I ceiling. **Abundance** and
**doublet-spatial** are reused from `final_sweep/scvi_baselines.npz`; the **triplet-spatial**
specialist is trained here with the same recipe (`n_latent=20, n_hidden=128, n_layers=2,
gene_likelihood='normal'`, inputs shifted ≥0, 200 epochs) and cached.

In [ ]:
import anndata as adm
import scvi

bl = np.load(FINAL / 'scvi_baselines.npz')
adata.obsm['z_scvi_abundance'] = bl['z_scvi_abundance']   # abundance ceiling
adata.obsm['z_scvi_spatial']   = bl['z_scvi_spatial']     # doublet-spatial ceiling

TRIP_NPZ = TRIP_OUT / 'scvi_triplet.npz'
if not TRIP_NPZ.exists():
    print('training triplet-spatial scVI specialist (GPU)…')
    scvi.settings.seed = 0
    Xd = np.asarray(get_dense(adata.obsm[TRIPLET_REP]), dtype='float32')
    Xd = np.clip(Xd - Xd.min(axis=0, keepdims=True), 0.0, None)     # scVI needs non-negative input
    a = adm.AnnData(X=Xd, obs=adata.obs.copy())
    a.var_names = [str(c) for c in adata.obsm[TRIPLET_REP].columns]
    scvi.model.SCVI.setup_anndata(a, batch_key=None)
    m = scvi.model.SCVI(a, n_latent=20, n_hidden=128, n_layers=2, dropout_rate=0.1, gene_likelihood='normal')
    m.train(max_epochs=200, batch_size=512, early_stopping=True)
    TRIP_OUT.mkdir(parents=True, exist_ok=True)
    np.savez(TRIP_NPZ, z_scvi_triplet=m.get_latent_representation())
    print('saved', TRIP_NPZ)

adata.obsm['z_scvi_triplet'] = np.load(TRIP_NPZ)['z_scvi_triplet']   # triplet-spatial ceiling
print('anchors:', {k: adata.obsm[k].shape for k in ['z_scvi_abundance', 'z_scvi_spatial', 'z_scvi_triplet']})

## Step 3 — load model latents + pairing

Doublet arm = the 3 cached best runs; triplet arm = the 2 depth-matched runs. Each contributes a
joint latent (`z__`), an abundance latent (`zabn__`), and a spatial latent (`zspt__`). Pairing
matches on depth (`L2`/`L3`); both `L2` doublet runs map to the single `L2` triplet run.

In [ ]:
DOUBLET_RUNS = ['arch-S3_mvtcae__sp-ct100__nl20__L2__bm-ft',
                'arch-S3_mvtcae__sp-top500__nl20__L2__bm-ft',
                'arch-S3_mvtcae__sp-ct100__nl20__L3__bm-ft']
TRIPLET_RUNS = ['arch-S3_mvtcae__sp-triplet__nl20__L2__bm-ft',
                'arch-S3_mvtcae__sp-triplet__nl20__L3__bm-ft']
RUN_DIR = {**{r: FINAL / r for r in DOUBLET_RUNS}, **{r: TRIP_OUT / r for r in TRIPLET_RUNS}}
arm   = {**{r: 'doublet' for r in DOUBLET_RUNS}, **{r: 'triplet' for r in TRIPLET_RUNS}}
depth = {r: ('L2' if '__L2__' in r else 'L3') for r in RUN_DIR}

def _sp(r):
    return 'ct100' if 'ct100' in r else ('top500' if 'top500' in r else 'trip')
labels = {r: f"{'D' if arm[r]=='doublet' else 'T'}·{_sp(r)}·{depth[r]}" for r in RUN_DIR}

def load_latents(rid):
    z = np.load(RUN_DIR[rid] / 'latents.npz')
    adata.obsm[f'z__{rid}']    = z['z_joint']
    adata.obsm[f'zabn__{rid}'] = z['z_abn']
    adata.obsm[f'zspt__{rid}'] = z['z_spt']

available = [r for r in DOUBLET_RUNS + TRIPLET_RUNS if (RUN_DIR[r] / 'latents.npz').exists()]
for r in available:
    load_latents(r)

trip_of = {'L2': TRIPLET_RUNS[0], 'L3': TRIPLET_RUNS[1]}
pairing = pd.DataFrame([{'doublet_run': labels[r], 'triplet_run': labels[trip_of[depth[r]]],
                         'depth': depth[r], '_d': r, '_t': trip_of[depth[r]]} for r in DOUBLET_RUNS])
print(f'{len(available)} / {len(DOUBLET_RUNS)+len(TRIPLET_RUNS)} runs loaded')
missing = [r for r in DOUBLET_RUNS + TRIPLET_RUNS if r not in available]
print('missing (run Step 1 first):', missing if missing else 'none')
pairing[['doublet_run', 'triplet_run', 'depth']]

## Step 4 — per-modality latent quality (Moran's I + AUC vs specialist)

For each run: the abundance latent is scored on abundance features (ceiling `scvi_abundance`); the
spatial latent is scored on its **own** modality features — doublet runs on `DOUBLET_REP` (ceiling
`scvi_spatial`), triplet runs on `TRIPLET_REP` (ceiling `scvi_triplet`). AUC = P(a feature is better
autocorrelated than under the specialist); 0.5 ties the ceiling, >0.5 beats it — comparable across
arms despite different feature counts.

In [ ]:
from scipy.stats import mannwhitneyu

def morans_auc(ac_df, anchor):
    ref = ac_df.loc[ac_df['latent'] == anchor, 'morans'].to_numpy()
    return pd.Series({name: mannwhitneyu(g['morans'].to_numpy(), ref, alternative='two-sided').statistic
                             / (len(g) * len(ref))
                      for name, g in ac_df.groupby('latent')})

# abundance: shared rep + ceiling
abn_ac   = distr_autocorrelation_in_latent(
    adata, [f'zabn__{r}' for r in available] + ['z_scvi_abundance'],
    list(available) + ['scvi_abundance'], rep_key=ABUNDANCE_REP, pca_kwargs=PCA_KW)
abn_mean = abn_ac.groupby('latent')['morans'].mean()
abn_auc  = morans_auc(abn_ac, 'scvi_abundance')

# spatial: each arm on its own modality features, vs its own specialist
dbl = [r for r in available if arm[r] == 'doublet']
tri = [r for r in available if arm[r] == 'triplet']
dbl_ac = distr_autocorrelation_in_latent(adata, [f'zspt__{r}' for r in dbl] + ['z_scvi_spatial'],
                                         dbl + ['scvi_spatial'], rep_key=DOUBLET_REP, pca_kwargs=PCA_KW)
tri_ac = distr_autocorrelation_in_latent(adata, [f'zspt__{r}' for r in tri] + ['z_scvi_triplet'],
                                         tri + ['scvi_triplet'], rep_key=TRIPLET_REP, pca_kwargs=PCA_KW)
spt_mean = pd.concat([dbl_ac, tri_ac]).groupby('latent')['morans'].mean()
spt_auc  = pd.concat([morans_auc(dbl_ac, 'scvi_spatial'), morans_auc(tri_ac, 'scvi_triplet')])

per_mod = pd.DataFrame({
    'label': pd.Series(labels), 'arm': pd.Series(arm), 'depth': pd.Series(depth),
    'abundance': abn_mean.reindex(available), 'abundance_auc': abn_auc.reindex(available),
    'spatial': spt_mean.reindex(available),   'spatial_auc': spt_auc.reindex(available),
}).loc[available]
print('ceilings  abn:', round(abn_mean['scvi_abundance'], 4),
      '| dbl-spatial:', round(spt_mean['scvi_spatial'], 4),
      '| trip-spatial:', round(spt_mean['scvi_triplet'], 4))
per_mod.round(4)

In [ ]:
# Autocorrelation histograms on the JOINT latent (z__{run}): how well the fused representation
# preserves each feature set. Abundance scored on ABUNDANCE_REP; spatial overlays doublet-arm joint
# latents on DOUBLET_REP and triplet-arm joint latents on TRIPLET_REP (each vs its own ceiling).
def _lab(n):
    return labels.get(n, n)                       # run-id -> short label; anchors keep their name

jabn_ac = distr_autocorrelation_in_latent(
    adata, [f'z__{r}' for r in available] + ['z_scvi_abundance'],
    list(available) + ['scvi_abundance'], rep_key=ABUNDANCE_REP, pca_kwargs=PCA_KW)
jdbl_ac = distr_autocorrelation_in_latent(
    adata, [f'z__{r}' for r in dbl] + ['z_scvi_spatial'],
    dbl + ['scvi_spatial'], rep_key=DOUBLET_REP, pca_kwargs=PCA_KW)
jtri_ac = distr_autocorrelation_in_latent(
    adata, [f'z__{r}' for r in tri] + ['z_scvi_triplet'],
    tri + ['scvi_triplet'], rep_key=TRIPLET_REP, pca_kwargs=PCA_KW)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- abundance (joint latent) ---
d = jabn_ac.assign(name=jabn_ac['latent'].map(_lab))
order = [_lab(n) for n in jabn_ac['latent'].unique() if n != 'scvi_abundance'] + ['scvi_abundance']
sns.histplot(data=d, x='morans', hue='name', hue_order=order, kde=True,
             stat='density', common_norm=False, alpha=0.35, ax=axes[0])
axes[0].axvline(jabn_ac.loc[jabn_ac['latent'] == 'scvi_abundance', 'morans'].mean(),
                color='k', ls='--', lw=1)
axes[0].set(title="joint latent — abundance Moran's I (dashed = scvi_abundance)", xlabel="Moran's I")

# --- spatial (joint latent): doublet + triplet on the same axis, each vs its own ceiling ---
spt_all = pd.concat([jdbl_ac, jtri_ac], ignore_index=True)
ceils = ['scvi_spatial', 'scvi_triplet']
d = spt_all.assign(name=spt_all['latent'].map(_lab))
order = [_lab(n) for n in spt_all['latent'].unique() if n not in ceils] + ceils
sns.histplot(data=d, x='morans', hue='name', hue_order=order, kde=True,
             stat='density', common_norm=False, alpha=0.35, ax=axes[1])
for c, col in zip(ceils, ['#4c72b0', '#c44e52']):
    axes[1].axvline(spt_all.loc[spt_all['latent'] == c, 'morans'].mean(), color=col, ls='--', lw=1.2)
axes[1].set(title="joint latent — spatial Moran's I: doublet vs triplet (dashed = ceilings)",
            xlabel="Moran's I")

for ax in axes:
    if ax.get_legend():
        sns.move_legend(ax, 'upper right', fontsize=6, title=None)
plt.tight_layout(); plt.show()

## Step 5 — cross-modality capture (joint latent on BOTH feature sets)

The crux of doublet-vs-triplet: score every run's **joint** latent on both `DOUBLET_REP` and
`TRIPLET_REP`. A good triplet model should raise triplet-feature Moran's I (its target) without
collapsing doublet-feature Moran's I (structure it never saw) — and vice versa. Anchors give each
feature set's ceiling (`scvi_spatial` for doublet feats, `scvi_triplet` for triplet feats).

In [ ]:
joint_keys  = [f'z__{r}' for r in available] + ['z_scvi_spatial', 'z_scvi_triplet']
joint_names = list(available)                + ['scvi_spatial', 'scvi_triplet']
jd = distr_autocorrelation_in_latent(adata, joint_keys, joint_names, rep_key=DOUBLET_REP,
                                     pca_kwargs=PCA_KW).groupby('latent')['morans'].mean()
jt = distr_autocorrelation_in_latent(adata, joint_keys, joint_names, rep_key=TRIPLET_REP,
                                     pca_kwargs=PCA_KW).groupby('latent')['morans'].mean()
cross = pd.DataFrame({'label': pd.Series(labels), 'arm': pd.Series(arm), 'depth': pd.Series(depth),
                      'joint_on_doublet': jd.reindex(available),
                      'joint_on_triplet': jt.reindex(available)}).loc[available]
print('ceilings  doublet-feat (scvi_spatial):', round(jd['scvi_spatial'], 4),
      '| triplet-feat (scvi_triplet):', round(jt['scvi_triplet'], 4))
cross.round(4).sort_values('joint_on_triplet', ascending=False)

## Step 6 — scib-metrics (bio conservation + batch correction)

Score joint + per-modality latents of all runs and the three specialists. Two orientations, exactly
as the ref sweep: (bio = cell type, batch = system) and the swap.

In [ ]:
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

emb = {}
for r in available:
    emb[f'{labels[r]}·joint'] = f'z__{r}'
    emb[f'{labels[r]}·abn']   = f'zabn__{r}'
    emb[f'{labels[r]}·spt']   = f'zspt__{r}'
emb['scvi_abundance'] = 'z_scvi_abundance'
emb['scvi_spatial']   = 'z_scvi_spatial'
emb['scvi_triplet']   = 'z_scvi_triplet'
for name, key in emb.items():
    adata.obsm[name] = adata.obsm[key]

def run_scib(label_key, batch_key):
    bm = Benchmarker(adata, batch_key=batch_key, label_key=label_key,
                     embedding_obsm_keys=list(emb.keys()),
                     bio_conservation_metrics=BioConservation(),
                     batch_correction_metrics=BatchCorrection())
    bm.benchmark()
    return bm

TRIP_OUT.mkdir(parents=True, exist_ok=True)
bm_celltype = run_scib(BIO_KEY, BATCH_KEY)     # bio = cell type, batch = system
bm_system   = run_scib(BATCH_KEY, BIO_KEY)     # bio = system,    batch = cell type
bm_celltype.get_results(min_max_scale=False).to_csv(TRIP_OUT / 'scib_celltype.csv')
bm_system.get_results(min_max_scale=False).to_csv(TRIP_OUT / 'scib_system.csv')
print(f'{len(emb)} embeddings scored | bio = {BIO_KEY} (batch = {BATCH_KEY})')
bm_celltype.plot_results_table(min_max_scale=False)

## Step 7 — posterior predictive reconstruction (per modality)

Feature-level reconstruction Pearson per modality, from each run's `ppc_metrics.parquet`
(`calculate_metrics`). Abundance is shared; the Spatial column is doublet-Z for the doublet arm and
triplet-Z for the triplet arm.

In [ ]:
ppc = pd.concat([pd.read_parquet(RUN_DIR[r] / 'ppc_metrics.parquet') for r in available
                 if (RUN_DIR[r] / 'ppc_metrics.parquet').exists()], ignore_index=True)
ppc_feat = ppc[ppc['Level'] == 'Features']
ppc_pearson = ppc_feat.pivot_table(index='Model', columns='Modality', values='Pearson')
ppc_pearson.insert(0, 'label', ppc_pearson.index.map(labels))
ppc_pearson.insert(1, 'arm', ppc_pearson.index.map(arm))
print('feature-level reconstruction Pearson (per modality)')
ppc_pearson.round(4)

In [ ]:
# PPC composite (ref-notebook style): per-model Observed-vs-Generated CV scatter (Features + Cells
# rows) + Delta-CV violin + stats table. One figure per modality; columns = runs (short labels).
# For the Spatial figure each column is that run's own modality (doublet feats for D·, triplet for T·).
from PixelGen.utils import plot_composite_ppc

ppc_lab = ppc.copy()
ppc_lab['Model'] = ppc_lab['Model'].map(lambda m: labels.get(m, m))
order = [labels[r] for r in available]
plot_composite_ppc('Abundance', ppc_lab, 'cornflowerblue', order); plt.show()
plot_composite_ppc('Spatial',   ppc_lab, 'lightgreen',     order); plt.show()

## Step 8 — doublet vs triplet summary

Pair each doublet best with its depth-matched triplet run and contrast the key metrics.

In [ ]:
M = per_mod.join(cross[['joint_on_doublet', 'joint_on_triplet']])
rows = []
for _, pr in pairing.iterrows():
    d, t = pr['_d'], pr['_t']
    if d not in M.index or t not in M.index:
        continue
    rows.append({
        'pair': f"{labels[d]}  vs  {labels[t]}",
        'spatial_auc_D': M.loc[d, 'spatial_auc'], 'spatial_auc_T': M.loc[t, 'spatial_auc'],
        'joint_tripfeat_D': M.loc[d, 'joint_on_triplet'], 'joint_tripfeat_T': M.loc[t, 'joint_on_triplet'],
        'joint_dblfeat_D':  M.loc[d, 'joint_on_doublet'], 'joint_dblfeat_T':  M.loc[t, 'joint_on_doublet'],
    })
cmp = pd.DataFrame(rows)
display(cmp.round(4))

if len(cmp):
    x = np.arange(len(cmp)); w = 0.38
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].bar(x - w/2, cmp['spatial_auc_D'], w, label='doublet', color='#4c72b0')
    axes[0].bar(x + w/2, cmp['spatial_auc_T'], w, label='triplet', color='#c44e52')
    axes[0].axhline(0.5, ls='--', lw=.8, color='k')
    axes[0].set_xticks(x); axes[0].set_xticklabels(cmp['pair'], rotation=30, ha='right', fontsize=7)
    axes[0].set(ylabel='spatial AUC vs specialist', title='Spatial latent quality: doublet vs triplet')
    axes[0].legend(fontsize=8)
    axes[1].bar(x - w/2, cmp['joint_tripfeat_D'], w, label='doublet arm', color='#4c72b0')
    axes[1].bar(x + w/2, cmp['joint_tripfeat_T'], w, label='triplet arm', color='#c44e52')
    axes[1].set_xticks(x); axes[1].set_xticklabels(cmp['pair'], rotation=30, ha='right', fontsize=7)
    axes[1].set(ylabel="joint Moran's I on triplet feats", title='Joint capture of triplet structure')
    axes[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()

## Methods note
- **Data**: `adata_cytovi_annotated_compat.h5ad` (15782 cells × 159 proteins). Abundance modality =
  `arcsinh` layer; batch = `cell_system`; bio labels = `cell_type_annot`.
- **Spatial modalities**: *doublet* = marker-pair join-count Z (`spatial_asinh5_top500var`, the
  ref-sweep spatial rep; per-run models train on their own `ct100`/`top500` selection). *triplet* =
  open-wedge Z over 171 curated triplets, `coloc_triplet_asinh5 = arcsinh(z/5)` (built by
  `utils.build_triplet_asinh5_obsm` from `results/chunglu/*_triplets_percell.parquet`).
- **Models**: the 3 best from `final_sweep_benchmark.ipynb` — all `S3_mvtcae` (POE + MVTCAE,
  `tc_alpha=0.5, tc_beta=1.0`), `n_latent=20`, batch-mask `ft`. Doublet arm reuses cached
  `final_sweep` artifacts; triplet arm = depth-matched `sp-triplet L2/L3` via the same driver
  (`train_final_sweep.py --spatial triplet`).
- **Benchmark** (identical to the ref sweep): per-modality Moran's I of the modality features in
  each latent's kNN graph + Mann-Whitney AUC vs the matching vanilla-scVI specialist (abundance,
  doublet-spatial, triplet-spatial); joint-latent Moran's I on **both** feature sets; scib-metrics
  (bio-conservation + batch-correction, both label/batch orientations); feature-level PPC Pearson
  from `calculate_metrics`.
- **Comparability**: AUC is vs each modality's own specialist ceiling, so doublet and triplet
  spatial scores share a 0–1 scale despite different feature counts. Cross-modality Moran's I (joint
  latent on the *other* modality's features) tests whether triplets add structure without discarding
  doublet structure.